# Bonus: RAG

Retrieval-augmented generation — ground agent answers in your own documents via a vector store / retriever tool.

**Key learnings**
- Retrieve relevant chunks before generation
- Vector stores ground answers in your documents
- Agents can call retrieval as a tool



### Load environment variables

`load_dotenv()` initializes the environment by loading the `.env` file so the OpenAI API key (used for embeddings and the model) is available.

**Key learnings**
- Retrieve relevant chunks before generation
- Vector stores ground answers in your documents
- Agents can call retrieval as a tool


In [ ]:
from dotenv import load_dotenv

load_dotenv()

## Semantic search

### Load the PDF document

`PyPDFLoader` reads the ACME employee handbook PDF and `loader.load()` returns it as a list of `Document` objects (text plus metadata). This is the raw source we'll index for retrieval.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("resources/acmecorp-employee-handbook.pdf")

data = loader.load()

print(data)

### Split the document into chunks

`RecursiveCharacterTextSplitter` breaks the document into overlapping chunks (1000 chars with 200 overlap). Chunking keeps each piece small enough to embed and retrieve precisely; the overlap avoids cutting context at chunk boundaries. `add_start_index=True` records where each chunk began in the original text.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(data)

print(len(all_splits))

Embedding Models: https://docs.langchain.com/oss/python/integrations/text_embedding

### Create the embedding model

`OpenAIEmbeddings` (model `text-embedding-3-large`) turns text into vectors. Similar meanings map to nearby vectors, which is what makes semantic search work.

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

### Create an in-memory vector store

`InMemoryVectorStore` holds the embedded chunks in memory and supports similarity search. It's simple and disposable, good for a demo (no external database needed).

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

### Add the chunks to the vector store

`add_documents` embeds each chunk with the embedding model and stores the vectors, returning their ids. After this the handbook is searchable by meaning.

In [ ]:
ids = vector_store.add_documents(documents=all_splits)

### Run a semantic search

`similarity_search` embeds the question and returns the chunks whose vectors are closest to it. We print the top result, which should be the PTO/vacation section.

In [ ]:
results = vector_store.similarity_search(
    "How many days of vacation does an employee get in their first year?"
)

print(results[0])

## RAG Agent

### Wrap retrieval as a tool

`search_handbook` exposes the vector store to an agent: given a query, it runs a similarity search and returns the most relevant chunk's text. This is the retrieval half of Retrieval-Augmented Generation (RAG).

In [ ]:
from langchain.tools import tool

@tool
def search_handbook(query: str) -> str:
    """Search the employee handbook for information"""
    results = vector_store.similarity_search(query)
    return results[0].page_content

### Create the RAG agent

`create_agent` builds an agent with the `search_handbook` tool and a system prompt telling it to answer questions by searching the handbook. The model decides when to call the tool and grounds its answer in the retrieved text.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[search_handbook],
    system_prompt="You are a helpful agent that can search the employee handbook for information."
    )

### Ask the agent a question

We ask how many vacation days a first-year employee gets. The agent calls `search_handbook`, reads the retrieved PTO policy, and composes an answer from it.

In [ ]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many days of vacation does an employee get in their first year?")]}
)

### Inspect the response

`pprint` shows the full message history: the question, the tool call, the retrieved handbook text, and the agent's grounded final answer.

In [ ]:
from pprint import pprint

pprint(response)